# NZ Lotto Powerball — XGBoost Training

This notebook trains two XGBoost models on your historical draw data:
- **Main model** (40 binary classifiers, one per number 1–40)
- **Powerball model** (10 binary classifiers, one per PB value 1–10)

**Steps:**
1. Run Cell 1 to install dependencies
2. Run Cell 2 to upload `lotto_draws.csv` (I'll export it for you)
3. Run Cell 3 to train (~5 minutes on CPU)
4. Run Cell 4 to download `model.pkl`

Uses only stdlib `csv` module to load data — no sqlite3 dependency.

In [ ]:
# Cell 1: Install XGBoost (only dependency — CSV loader uses stdlib)
!pip install xgboost numpy -q
print('Ready')

In [ ]:
# Cell 2: Upload DB or CSV
# You can upload either lotto_working.db OR the pre-exported lotto_draws.csv
from google.colab import files
print("Select lotto_draws.csv (or lotto_working.db)...")
uploaded = files.upload()

import os
for name in uploaded:
    size = os.path.getsize(name) / 1024
    print(f'Uploaded: {name} ({size:.0f} KB)')

In [ ]:
# Cell 3: Train the model
# Loads from CSV (no sqlite3 dependency) — uses only stdlib + numpy + xgboost

from __future__ import annotations

import csv
import os
import pickle
import sys
import warnings
from collections import Counter
from typing import Any

import numpy as np
import xgboost as xgb

warnings.filterwarnings('ignore', category=UserWarning)

# --- Data loading from CSV ---
def load_draws(path='lotto_draws.csv'):
    """Load draws from CSV file (no sqlite3 needed)."""
    with open(path) as f:
        reader = csv.DictReader(f)
        draws = []
        for row in reader:
            nums = [int(row[f'n{i}']) for i in range(1, 7)]
            pb = int(row['powerball'])
            date = row['draw_date']
            draws.append((nums, pb, date))
    return draws

# --- Feature engineering (17 features per number) ---
MIN_HISTORY = 30

def build_features(draws, target_idx, num_range):
    """Build feature vectors for one draw, one per number in num_range."""
    if target_idx < MIN_HISTORY:
        return []

    past = draws[:target_idx]
    current = draws[target_idx][0]
    is_pb = num_range[0] == 1 and num_range[-1] == 10
    if is_pb:
        current = [draws[target_idx][1]]

    freq_10, freq_30, freq_all = Counter(), Counter(), Counter()
    last_appear = {}
    lag_1, lag_2, lag_3 = set(), set(), set()
    pos_sums, pos_n = {}, {}
    pos_counts = {n: Counter() for n in num_range}
    cooccur = {n: Counter() for n in num_range}
    streak_cur, streak_max = {}, {}
    for n in num_range:
        streak_cur[n] = 0
        streak_max[n] = 0
        pos_sums[n] = 0.0
        pos_n[n] = 0

    recent_50 = max(0, target_idx - 50)

    for j, (nums, pb, _) in enumerate(past):
        drawn = set(nums)
        if is_pb:
            drawn = {pb}
        for n in num_range:
            hit = n in drawn
            if j >= target_idx - 10 and hit:
                freq_10[n] += 1
            if j >= target_idx - 30 and hit:
                freq_30[n] += 1
            if hit:
                freq_all[n] += 1
            if j == target_idx - 1 and hit:
                lag_1.add(n)
            if j == target_idx - 2 and hit:
                lag_2.add(n)
            if j == target_idx - 3 and hit:
                lag_3.add(n)
            if hit:
                last_appear[n] = j
            if hit:
                streak_cur[n] += 1
                streak_max[n] = max(streak_max[n], streak_cur[n])
            else:
                streak_cur[n] = 0
            if not is_pb and hit and n in nums:
                pos = sorted(nums).index(n)
                pos_sums[n] += pos
                pos_n[n] += 1
                pos_counts[n][pos] += 1
            if j >= recent_50 and hit:
                for other in nums:
                    if other != n and other in num_range:
                        cooccur[n][other] += 1
                if is_pb and n == pb:
                    for other in nums:
                        if other in num_range:
                            cooccur[n][other] += 1

    results = []
    for n in num_range:
        gap = target_idx - last_appear.get(n, 0) - 1 if n in last_appear else target_idx
        gap = min(gap, 500)
        total = max(len(past), 1)
        avg_pos = pos_sums[n] / pos_n[n] if pos_n[n] > 0 else -1.0
        hot = [x for x, _ in freq_30.most_common(3)]
        co_hot = sum(cooccur[n].get(h, 0) for h in hot)
        mcp = pos_counts[n].most_common(1)[0][0] if pos_counts[n] else -1

        features = [
            n / (num_range[-1] + 1),
            float(freq_10.get(n, 0)),
            float(freq_30.get(n, 0)),
            float(freq_all.get(n, 0)),
            gap / 500.0,
            1.0 if n in lag_1 else 0.0,
            1.0 if n in lag_2 else 0.0,
            1.0 if n in lag_3 else 0.0,
            freq_10.get(n, 0) / max(target_idx - max(0, target_idx - 10), 1),
            freq_30.get(n, 0) / 30.0,
            freq_all.get(n, 0) / total,
            avg_pos / 5.0,
            float(mcp),
            float(streak_max[n]),
            1.0 if n <= 10 else (2.0 if n <= 20 else (3.0 if n <= 30 else 4.0)),
            1.0 if n % 2 == 1 else 0.0,
            min(co_hot / 50.0, 1.0),
        ]
        label = 1 if n in current else 0
        results.append({'features': features, 'label': label, 'num': n})
    return results

# --- Dataset creation (chronological split) ---
def create_dataset(draws, num_range, test_split=0.2):
    all_rows = []
    for i in range(MIN_HISTORY, len(draws)):
        all_rows.extend(build_features(draws, i, num_range))
    X = np.array([r['features'] for r in all_rows], dtype=np.float32)
    y = np.array([r['label'] for r in all_rows], dtype=np.int32)
    n_nums = len(num_range)
    n_draws = len(all_rows) // n_nums
    split = int(n_draws * (1 - test_split)) * n_nums
    return X[:split], X[split:], y[:split], y[split:]

# --- AUC (pure numpy, no sklearn) ---
def roc_auc(y_true, y_score):
    n_pos, n_neg = int(y_true.sum()), len(y_true) - int(y_true.sum())
    if n_pos == 0 or n_neg == 0:
        return 0.5
    order = np.argsort(y_score)
    rank_sum = np.sum(np.where(y_true[order] == 1)[0]) + n_pos
    return float((rank_sum - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg))

# --- Train ---
def train_model(X_train, X_test, y_train, y_test, label):
    sw = (len(y_train) - y_train.sum()) / max(y_train.sum(), 1)
    print(f'  scale_pos_weight={sw:.2f}')
    model = xgb.XGBClassifier(
        n_estimators=500, max_depth=6, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=sw, eval_metric='logloss',
        random_state=42, verbosity=0,
    )
    model.fit(X_train, y_train,
              eval_set=[(X_test, y_test)],
              early_stopping_rounds=30, verbose=False)

    train_auc = roc_auc(y_train, model.predict_proba(X_train)[:, 1])
    test_auc = roc_auc(y_test, model.predict_proba(X_test)[:, 1])
    best = getattr(model, 'best_iteration', model.get_params()['n_estimators'])
    print(f'  Best iteration: {best}')
    print(f'  Train AUC:      {train_auc:.4f}')
    print(f'  Test AUC:       {test_auc:.4f}')

    feat_names = [
        'norm_num', 'freq_10', 'freq_30', 'freq_all',
        'norm_gap', 'lag_1', 'lag_2', 'lag_3',
        'roll_mean_10', 'roll_mean_30', 'overall_rate',
        'norm_avg_pos', 'most_common_pos', 'max_streak',
        'decade', 'is_odd', 'cooccur_hot',
    ]
    top5 = np.argsort(model.feature_importances_)[::-1][:5]
    print('  Top 5 features:')
    for i in top5:
        print(f'    {feat_names[i]:>16s}  {model.feature_importances_[i]:.4f}')
    return model

# --- Predict next draw ---
def predict_next(draws, main_model, pb_model):
    nxt = len(draws)
    main_rows = build_features(draws, nxt, range(1, 41))
    X = np.array([r['features'] for r in main_rows], dtype=np.float32)
    probs = main_model.predict_proba(X)[:, 1]
    pairs = sorted([(r['num'], probs[i]) for i, r in enumerate(main_rows)],
                   key=lambda x: x[1], reverse=True)
    top6 = sorted([n for n, _ in pairs[:6]])
    top_probs = {n: p for n, p in pairs[:6]}

    pb_rows = build_features(draws, nxt, range(1, 11))
    if pb_model and pb_rows:
        X_pb = np.array([r['features'] for r in pb_rows], dtype=np.float32)
        probs_pb = pb_model.predict_proba(X_pb)[:, 1]
        pb_pairs = sorted([(r['num'], probs_pb[i]) for i, r in enumerate(pb_rows)],
                         key=lambda x: x[1], reverse=True)
        pb, pb_prob = pb_pairs[0]
    else:
        pb = Counter(pb for _, pb, _ in draws[-30:]).most_common(1)[0][0]
        pb_prob = 0.0

    return {'numbers': top6, 'powerball': pb,
            'number_probs': top_probs, 'pb_prob': pb_prob}

# --- Main ---
print('Loading draws...')
draws = load_draws()
print(f'  Loaded {len(draws)} draws ({draws[0][2]} to {draws[-1][2]})')

print('\n--- Main Numbers (1-40) ---')
X_tr, X_te, y_tr, y_te = create_dataset(draws, range(1, 41))
print(f'  Train: {X_tr.shape[0]}  Test: {X_te.shape[0]}  '
      f'Positive: {y_tr.mean():.2%} / {y_te.mean():.2%}')
main_model = train_model(X_tr, X_te, y_tr, y_te, 'main')

print('\n--- Powerball (1-10) ---')
X_pt, X_pe, y_pt, y_pe = create_dataset(draws, range(1, 11))
print(f'  Train: {X_pt.shape[0]}  Test: {X_pe.shape[0]}  '
      f'Positive: {y_pt.mean():.2%} / {y_pe.mean():.2%}')
pb_model = train_model(X_pt, X_pe, y_pt, y_pe, 'powerball')

print('\n--- Prediction for Next Draw ---')
pred = predict_next(draws, main_model, pb_model)
nums = ', '.join(f'{n:02d}' for n in pred['numbers'])
probs = ', '.join(f'#{n}: {pred["number_probs"][n]:.1%}' for n in pred['numbers'])
print(f'  Numbers:      {nums}')
print(f'  Probs:        {probs}')
print(f'  Powerball:    {pred["powerball"]}  (prob: {pred["pb_prob"]:.1%})')

# --- Save ---
model_data = {
    'main_model': main_model,
    'pb_model': pb_model,
    'draws_used': len(draws),
    'date_range': (draws[0][2], draws[-1][2]),
    'test_auc': roc_auc(y_te, main_model.predict_proba(X_te)[:, 1]),
}
with open('model.pkl', 'wb') as f:
    pickle.dump(model_data, f, protocol=pickle.HIGHEST_PROTOCOL)
size = os.path.getsize('model.pkl') / 1024
print(f'\nModel saved to model.pkl ({size:.0f} KB)')

In [ ]:
# Cell 4: Download model
from google.colab import files
files.download('model.pkl')
print('\nAfter downloading:')
print('  1. Place model.pkl in the lotto-wheel-app directory')
print('  2. Run: python3 predict_ml.py')